In [1]:
import numpy as np
import matplotlib.pyplot as plt
import emcee
import matplotlib
matplotlib.use("Agg")

In [2]:
#an implementation of the berryscm function
def berryscm(k, mu, asp, x, ro1, P_water):

    k, mu, asp, x = map(np.asarray, (k, mu, asp, x))
    n = len(k)

    asp[asp == 1] = 0.99

    ksc = np.sum(k * x)
    musc = np.sum(mu * x)
    tol = 1e-6 * k[0]
    max_iter = 3000
    n_iter = 0
    knew = 0
    del_k = np.abs(ksc-knew)


    theta = np.zeros_like(asp)
    fn = np.zeros_like(asp)
    obdx = asp < 1
    prdx = asp > 1

    theta[obdx] = (asp[obdx] / (1 - asp[obdx]**2)**1.5) * (np.arccos(asp[obdx]) - asp[obdx] * np.sqrt(1 - asp[obdx]**2))
    fn[obdx] = (asp[obdx]**2 / (1 - asp[obdx]**2)) * (3 * theta[obdx] - 2)

    theta[prdx] = (asp[prdx] / (asp[prdx]**2 - 1)**1.5) * (asp[prdx] * np.sqrt(asp[prdx]**2 - 1) - np.arccosh(asp[prdx]))
    fn[prdx] = (asp[prdx]**2 / (asp[prdx]**2 - 1)) * (2 - 3 * theta[prdx])

    while del_k > np.abs(tol) and n_iter < max_iter:
        nu = (3 * ksc - 2 * musc) / (2 * (3 * ksc + musc))
        a = mu / musc - 1
        b = (1/3) * (k / ksc - mu / musc)
        r = (1 - 2 * nu) / (2 * (1 - nu))

        f1 = 1 + a * ((3/2) * (fn + theta) - r * ((3/2) * fn + (5/2) * theta - (4/3)))
        f2 = 1 + a * (1 + (3/2) * (fn + theta) - (r/2) * (3 * fn + 5 * theta)) + \
             b * (3 - 4 * r)

        f2 += (a / 2) * (a + 3 * b) * (3 - 4 * r) * (fn + theta - r * (fn - theta + 2 * theta**2))

        f3 = 1 + a * (1 - (fn + (3 / 2) * theta) + r * (fn + theta))
        f4 = 1 + (a / 4) * (fn + 3 * theta - r * (fn - theta))
        f5 = a * (-fn + r * (fn + theta - (4 / 3))) + b * theta * (3 - 4 * r)
        f6 = 1 + a * (1 + fn - r * (fn + theta)) + b * (1 - theta) * (3 - 4 * r)
        f7 = 2 + (a / 4) * (3 * fn + 9 * theta - r * (3 * fn + 5 * theta)) + b * theta * (3 - 4 * r)
        f8 = a * (1 - 2 * r + (fn / 2) * (r - 1) + (theta / 2) * (5 * r - 3)) + \
         b * (1 - theta) * (3 - 4 * r)
        f9 = a * ((r - 1) * fn - r * theta) + b * theta * (3 - 4 * r)

        p = 3 * f1 / f2
        q = (2 / f3) + (1 / f4) + ((f4 * f5 + f6 * f7 - f8 * f9) / (f2 * f4))

        p /= 3
        q /= 5

        knew = np.sum(x * k * p) / np.sum(x * p)
        munew = np.sum(x * mu * q) / np.sum(x * q)

        del_k = np.abs(ksc - knew)
        ksc = knew
        musc = munew
        n_iter += 1

    kbr, mubr = ksc, musc

    rofl1 = 0.020
    ro_water = 1000
    k_water = 2.2e9
    rofl2 = P_water * ro_water + (1 - P_water) * rofl1
    kfl1 = 0.015
    kfl3 = 1/((1-P_water)/kfl1+P_water/k_water)
    kfl4 = (1-P_water)*kfl1+P_water*k_water
    kfl2 = (kfl3 + kfl4)/2
    phi = x[1]

    a = kbr / (k[0] - kbr) - kfl1 / (phi * (k[0] - kfl1)) + kfl2 / (phi * (k[0] - kfl2))
    k2 = k[0] * a / (1 + a)
    ro2 = ro1 - phi * rofl1 + phi * rofl2

    vp = np.sqrt((k2 + (4/3) * mubr) / ro2)
    vs = np.sqrt(mubr / ro2)

    return kbr, mubr, vp, vs, ro2, k2

In [3]:
#calculate vp, vs,rhob and other parameters from the randomly generated model model parameters
def myberry(theta):
    asp = theta[0]
    x_phi = theta[1]
    rock_vol = 1-x_phi
    x = np.array([rock_vol, x_phi])
    rock_density = theta[-1]*rock_vol
    gas_density = 0.02*x_phi
    rhob1 = rock_density + gas_density
    k = np.array([theta[3]*1e9, 0])
    mu = np.array([theta[4]*1e9, 0])
    P_water = theta[2]
    kbr, mubr, vp, vs, ro2, k2 = berryscm(k, mu, asp, x, rhob1, P_water)
    out = np.array([vp/(1e3), vs/(1e3), ro2])
    return out

In [4]:
def log_post(theta, lb, ub, d, s, H, prior_sig=0, prior_mu=0, gs=False):
    # 1) Always compute dM first
    dM = myberry(theta)           # shape (3,) for [vp, vs, rho]

    # 2) Strict bound check (lb < θ ≤ ub)
    if np.any(theta <= lb) or np.any(theta > ub):
        return -np.inf, dM        # <-- return dM, not None

    # 3) Physics constraints (vp, vs, rho)
    nH = np.sum(H)
    vp, vs, rho = dM
    if   (nH == 2 and H[0] and H[1] and not (vp>vs)) \
      or (nH == 3            and not (vp>vs)) \
      or (nH == 2 and (H[0] or H[1]) and not (2500 < rho < 3100)) \
      or (nH == 1 and H[2]             and not (2500 < rho < 3100)):
        return -np.inf, dM        # <-- still returns dM

    # 4) Misfit
    misfit = -0.5*np.sum(((d - dM)/s)**2)

    # 5) Optional Gaussian prior
    prior = -0.5*np.sum(((theta-prior_mu)/prior_sig)**2) if gs else 0

    # 6) Final return
    logp = misfit + prior
    return logp, dM

In [ ]:
lb = np.array([0, 0, 0, 75.6, 5, 2680])
ub = np.array([1, 0.5, 1, 107.6, 76.8, 4250])
n = np.shape(ub)[0]
H = np.array([1,1,1], dtype=int)
Ne = 3*n
prior_pdf = np.random.uniform(lb, ub, (Ne, n))
d = np.array([4.1, 2.5, 2537])
s = np.array([0.2, 0.3, 167])
sampler = emcee.EnsembleSampler(Ne,n,log_post,args=(lb, ub, d, s, H))
Nsteps = 10000
sampler.run_mcmc(prior_pdf, Nsteps, progress=True)
blobs = sampler.get_blobs()

# Extract samples and analyze results
samples = sampler.get_chain(flat=True)

# Plot results
labels = ["Aspect Ratio", "Porosity", "Water Content", "Bulk Modulus", "Shear Modulus", "Density"]
fig, axes = plt.subplots(n, figsize=(10, 7), sharex=True)
for i in range(n):
    axes[i].plot(samples[: , i], "k", alpha=0.3)
    axes[i].set_ylabel(labels[i])
axes[-1].set_xlabel("Step Number")
plt.tight_layout()
plt.show()

  0%|                                                  | 0/5000 [00:00<?, ?it/s]/var/folders/5p/cy9rmpp11w7ddk5wt_dw0lk00000gp/T/ipykernel_81020/2875182742.py:81: RuntimeWarning: invalid value encountered in sqrt
  vs = np.sqrt(mubr / ro2)
  0%|                                          | 2/5000 [00:00<05:12, 15.97it/s]/var/folders/5p/cy9rmpp11w7ddk5wt_dw0lk00000gp/T/ipykernel_81020/2875182742.py:80: RuntimeWarning: invalid value encountered in sqrt
  vp = np.sqrt((k2 + (4/3) * mubr) / ro2)
  0%|                                          | 4/5000 [00:00<05:39, 14.74it/s]/var/folders/5p/cy9rmpp11w7ddk5wt_dw0lk00000gp/T/ipykernel_81020/2875182742.py:80: RuntimeWarning: invalid value encountered in sqrt
  vp = np.sqrt((k2 + (4/3) * mubr) / ro2)
  0%|                                         | 13/5000 [00:00<04:05, 20.34it/s]/var/folders/5p/cy9rmpp11w7ddk5wt_dw0lk00000gp/T/ipykernel_81020/2875182742.py:80: RuntimeWarning: invalid value encountered in sqrt
  vp = np.sqrt((k2 + (4/3) * mubr) / 

/var/folders/5p/cy9rmpp11w7ddk5wt_dw0lk00000gp/T/ipykernel_81020/2875182742.py:80: RuntimeWarning: invalid value encountered in sqrt
  vp = np.sqrt((k2 + (4/3) * mubr) / ro2)
/var/folders/5p/cy9rmpp11w7ddk5wt_dw0lk00000gp/T/ipykernel_81020/2875182742.py:81: RuntimeWarning: invalid value encountered in sqrt
  vs = np.sqrt(mubr / ro2)
  3%|█▏                                      | 144/5000 [00:19<14:12,  5.70it/s]/var/folders/5p/cy9rmpp11w7ddk5wt_dw0lk00000gp/T/ipykernel_81020/2875182742.py:80: RuntimeWarning: invalid value encountered in sqrt
  vp = np.sqrt((k2 + (4/3) * mubr) / ro2)
/var/folders/5p/cy9rmpp11w7ddk5wt_dw0lk00000gp/T/ipykernel_81020/2875182742.py:80: RuntimeWarning: invalid value encountered in sqrt
  vp = np.sqrt((k2 + (4/3) * mubr) / ro2)
  3%|█▏                                      | 148/5000 [00:19<09:47,  8.27it/s]/var/folders/5p/cy9rmpp11w7ddk5wt_dw0lk00000gp/T/ipykernel_81020/2875182742.py:80: RuntimeWarning: invalid value encountered in sqrt
  vp = np.sqrt((k2 + (

  4%|█▍                                      | 184/5000 [00:25<13:06,  6.12it/s]/var/folders/5p/cy9rmpp11w7ddk5wt_dw0lk00000gp/T/ipykernel_81020/2875182742.py:81: RuntimeWarning: invalid value encountered in sqrt
  vs = np.sqrt(mubr / ro2)
  4%|█▌                                      | 188/5000 [00:25<08:32,  9.39it/s]/var/folders/5p/cy9rmpp11w7ddk5wt_dw0lk00000gp/T/ipykernel_81020/2875182742.py:80: RuntimeWarning: invalid value encountered in sqrt
  vp = np.sqrt((k2 + (4/3) * mubr) / ro2)
/var/folders/5p/cy9rmpp11w7ddk5wt_dw0lk00000gp/T/ipykernel_81020/2875182742.py:81: RuntimeWarning: invalid value encountered in sqrt
  vs = np.sqrt(mubr / ro2)
/var/folders/5p/cy9rmpp11w7ddk5wt_dw0lk00000gp/T/ipykernel_81020/2875182742.py:80: RuntimeWarning: invalid value encountered in sqrt
  vp = np.sqrt((k2 + (4/3) * mubr) / ro2)
/var/folders/5p/cy9rmpp11w7ddk5wt_dw0lk00000gp/T/ipykernel_81020/2875182742.py:81: RuntimeWarning: invalid value encountered in sqrt
  vs = np.sqrt(mubr / ro2)
  4%|█▌   

  5%|██                                      | 257/5000 [00:34<07:18, 10.81it/s]/var/folders/5p/cy9rmpp11w7ddk5wt_dw0lk00000gp/T/ipykernel_81020/2875182742.py:80: RuntimeWarning: invalid value encountered in sqrt
  vp = np.sqrt((k2 + (4/3) * mubr) / ro2)
  5%|██                                      | 263/5000 [00:35<10:07,  7.80it/s]/var/folders/5p/cy9rmpp11w7ddk5wt_dw0lk00000gp/T/ipykernel_81020/2875182742.py:80: RuntimeWarning: invalid value encountered in sqrt
  vp = np.sqrt((k2 + (4/3) * mubr) / ro2)
  5%|██▏                                     | 269/5000 [00:35<07:53, 10.00it/s]/var/folders/5p/cy9rmpp11w7ddk5wt_dw0lk00000gp/T/ipykernel_81020/2875182742.py:81: RuntimeWarning: invalid value encountered in sqrt
  vs = np.sqrt(mubr / ro2)
  5%|██▏                                     | 271/5000 [00:36<08:54,  8.85it/s]/var/folders/5p/cy9rmpp11w7ddk5wt_dw0lk00000gp/T/ipykernel_81020/2875182742.py:80: RuntimeWarning: invalid value encountered in sqrt
  vp = np.sqrt((k2 + (4/3) * mubr) / 

  7%|██▉                                     | 369/5000 [00:53<24:52,  3.10it/s]/var/folders/5p/cy9rmpp11w7ddk5wt_dw0lk00000gp/T/ipykernel_81020/2875182742.py:80: RuntimeWarning: invalid value encountered in sqrt
  vp = np.sqrt((k2 + (4/3) * mubr) / ro2)
/var/folders/5p/cy9rmpp11w7ddk5wt_dw0lk00000gp/T/ipykernel_81020/2875182742.py:81: RuntimeWarning: invalid value encountered in sqrt
  vs = np.sqrt(mubr / ro2)
/var/folders/5p/cy9rmpp11w7ddk5wt_dw0lk00000gp/T/ipykernel_81020/2875182742.py:80: RuntimeWarning: invalid value encountered in sqrt
  vp = np.sqrt((k2 + (4/3) * mubr) / ro2)
/var/folders/5p/cy9rmpp11w7ddk5wt_dw0lk00000gp/T/ipykernel_81020/2875182742.py:81: RuntimeWarning: invalid value encountered in sqrt
  vs = np.sqrt(mubr / ro2)
  7%|██▉                                     | 370/5000 [00:54<28:29,  2.71it/s]/var/folders/5p/cy9rmpp11w7ddk5wt_dw0lk00000gp/T/ipykernel_81020/2875182742.py:80: RuntimeWarning: invalid value encountered in sqrt
  vp = np.sqrt((k2 + (4/3) * mubr) / 

...

 89%|██████████████████████████████████▊    | 4468/5000 [17:15<01:48,  4.89it/s]/var/folders/5p/cy9rmpp11w7ddk5wt_dw0lk00000gp/T/ipykernel_81020/2875182742.py:80: RuntimeWarning: invalid value encountered in sqrt
  vp = np.sqrt((k2 + (4/3) * mubr) / ro2)
 89%|██████████████████████████████████▊    | 4469/5000 [17:15<01:54,  4.65it/s]/var/folders/5p/cy9rmpp11w7ddk5wt_dw0lk00000gp/T/ipykernel_81020/2875182742.py:80: RuntimeWarning: invalid value encountered in sqrt
  vp = np.sqrt((k2 + (4/3) * mubr) / ro2)
/var/folders/5p/cy9rmpp11w7ddk5wt_dw0lk00000gp/T/ipykernel_81020/2875182742.py:81: RuntimeWarning: invalid value encountered in sqrt
  vs = np.sqrt(mubr / ro2)
 89%|██████████████████████████████████▉    | 4472/5000 [17:15<01:32,  5.71it/s]/var/folders/5p/cy9rmpp11w7ddk5wt_dw0lk00000gp/T/ipykernel_81020/2875182742.py:80: RuntimeWarning: invalid value encountered in sqrt
  vp = np.sqrt((k2 + (4/3) * mubr) / ro2)
/var/folders/5p/cy9rmpp11w7ddk5wt_dw0lk00000gp/T/ipykernel_81020/2875182742

 92%|███████████████████████████████████▊   | 4589/5000 [17:32<01:04,  6.35it/s]/var/folders/5p/cy9rmpp11w7ddk5wt_dw0lk00000gp/T/ipykernel_81020/2875182742.py:80: RuntimeWarning: invalid value encountered in sqrt
  vp = np.sqrt((k2 + (4/3) * mubr) / ro2)
 92%|███████████████████████████████████▊   | 4590/5000 [17:32<01:12,  5.65it/s]/var/folders/5p/cy9rmpp11w7ddk5wt_dw0lk00000gp/T/ipykernel_81020/2875182742.py:80: RuntimeWarning: invalid value encountered in sqrt
  vp = np.sqrt((k2 + (4/3) * mubr) / ro2)
/var/folders/5p/cy9rmpp11w7ddk5wt_dw0lk00000gp/T/ipykernel_81020/2875182742.py:81: RuntimeWarning: invalid value encountered in sqrt
  vs = np.sqrt(mubr / ro2)
 92%|███████████████████████████████████▊   | 4591/5000 [17:32<01:20,  5.08it/s]/var/folders/5p/cy9rmpp11w7ddk5wt_dw0lk00000gp/T/ipykernel_81020/2875182742.py:80: RuntimeWarning: invalid value encountered in sqrt
  vp = np.sqrt((k2 + (4/3) * mubr) / ro2)
/var/folders/5p/cy9rmpp11w7ddk5wt_dw0lk00000gp/T/ipykernel_81020/2875182742

 94%|████████████████████████████████████▊  | 4717/5000 [17:48<00:31,  9.05it/s]/var/folders/5p/cy9rmpp11w7ddk5wt_dw0lk00000gp/T/ipykernel_81020/2875182742.py:80: RuntimeWarning: invalid value encountered in sqrt
  vp = np.sqrt((k2 + (4/3) * mubr) / ro2)
/var/folders/5p/cy9rmpp11w7ddk5wt_dw0lk00000gp/T/ipykernel_81020/2875182742.py:81: RuntimeWarning: invalid value encountered in sqrt
  vs = np.sqrt(mubr / ro2)
 94%|████████████████████████████████████▊  | 4719/5000 [17:48<00:33,  8.32it/s]/var/folders/5p/cy9rmpp11w7ddk5wt_dw0lk00000gp/T/ipykernel_81020/2875182742.py:80: RuntimeWarning: invalid value encountered in sqrt
  vp = np.sqrt((k2 + (4/3) * mubr) / ro2)
 94%|████████████████████████████████████▊  | 4724/5000 [17:48<00:20, 13.37it/s]/var/folders/5p/cy9rmpp11w7ddk5wt_dw0lk00000gp/T/ipykernel_81020/2875182742.py:80: RuntimeWarning: invalid value encountered in sqrt
  vp = np.sqrt((k2 + (4/3) * mubr) / ro2)
/var/folders/5p/cy9rmpp11w7ddk5wt_dw0lk00000gp/T/ipykernel_81020/2875182742

 96%|█████████████████████████████████████▍ | 4795/5000 [17:57<00:24,  8.38it/s]/var/folders/5p/cy9rmpp11w7ddk5wt_dw0lk00000gp/T/ipykernel_81020/2875182742.py:81: RuntimeWarning: invalid value encountered in sqrt
  vs = np.sqrt(mubr / ro2)
/var/folders/5p/cy9rmpp11w7ddk5wt_dw0lk00000gp/T/ipykernel_81020/2875182742.py:81: RuntimeWarning: invalid value encountered in sqrt
  vs = np.sqrt(mubr / ro2)
 96%|█████████████████████████████████████▍ | 4796/5000 [17:58<00:35,  5.79it/s]/var/folders/5p/cy9rmpp11w7ddk5wt_dw0lk00000gp/T/ipykernel_81020/2875182742.py:80: RuntimeWarning: invalid value encountered in sqrt
  vp = np.sqrt((k2 + (4/3) * mubr) / ro2)
 96%|█████████████████████████████████████▍ | 4800/5000 [17:58<00:33,  5.95it/s]/var/folders/5p/cy9rmpp11w7ddk5wt_dw0lk00000gp/T/ipykernel_81020/2875182742.py:81: RuntimeWarning: invalid value encountered in sqrt
  vs = np.sqrt(mubr / ro2)
 96%|█████████████████████████████████████▍ | 4803/5000 [17:59<00:29,  6.62it/s]/var/folders/5p/cy9rmpp11

 98%|██████████████████████████████████████▎| 4910/5000 [18:15<00:14,  6.37it/s]/var/folders/5p/cy9rmpp11w7ddk5wt_dw0lk00000gp/T/ipykernel_81020/2875182742.py:80: RuntimeWarning: invalid value encountered in sqrt
  vp = np.sqrt((k2 + (4/3) * mubr) / ro2)
 98%|██████████████████████████████████████▎| 4914/5000 [18:15<00:16,  5.16it/s]/var/folders/5p/cy9rmpp11w7ddk5wt_dw0lk00000gp/T/ipykernel_81020/2875182742.py:80: RuntimeWarning: invalid value encountered in sqrt
  vp = np.sqrt((k2 + (4/3) * mubr) / ro2)
 98%|██████████████████████████████████████▎| 4917/5000 [18:16<00:13,  6.18it/s]/var/folders/5p/cy9rmpp11w7ddk5wt_dw0lk00000gp/T/ipykernel_81020/2875182742.py:80: RuntimeWarning: invalid value encountered in sqrt
  vp = np.sqrt((k2 + (4/3) * mubr) / ro2)
/var/folders/5p/cy9rmpp11w7ddk5wt_dw0lk00000gp/T/ipykernel_81020/2875182742.py:81: RuntimeWarning: invalid value encountered in sqrt
  vs = np.sqrt(mubr / ro2)
 98%|██████████████████████████████████████▎| 4918/5000 [18:16<00:14,  5.60

In [6]:
def plot_1d_histograms(samples, labels=None, bins=100, savefig=None):
    n_parameters = samples.shape[1]  
    if labels is None:
        labels = [f"Parameter {i+1}" for i in range(n_parameters)]
    
    fig, axes = plt.subplots(n_parameters, 1, figsize=(8, 2 * n_parameters), sharex=False)
    if n_parameters == 1:
        axes = [axes]
    
    for i in range(n_parameters):
        ax = axes[i]
        ax.hist(samples[:, i], bins=bins, density=True, color='skyblue', edgecolor='black', alpha=0.7)
        ax.set_ylabel("Density")
        ax.set_title(labels[i])
    axes[-1].set_xlabel("Parameter Value")
    
    plt.tight_layout()
    
    plt.show()
    
    if savefig:
        plt.savefig(savefig)
    

In [7]:
label = np.array(['aspect ratio', 'porosity', 'water saturation', 'mineral bulk modulus', 'mineral shear modulus', 'mineral density'])
plot_1d_histograms(samples, labels = label, bins=100, savefig = 'Bmw_fig1.png')

/var/folders/5p/cy9rmpp11w7ddk5wt_dw0lk00000gp/T/ipykernel_81020/964710158.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [8]:
param_names = list(label)

# Find the indices for porosity and water saturation
idx_porosity = param_names.index('porosity')
idx_water    = param_names.index('water saturation')

def plot_and_save(param_idx, param_name, minn, maxx, mayy, bins=100, filename=None):
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.hist(samples[:, param_idx], bins=bins, density=True, alpha=0.7, edgecolor='black')
    #ax.set_xlabel(param_name.capitalize())
    #ax.set_ylabel("Density")
    #ax.set_title(f"Histogram of {param_name.capitalize()}")
    ax.set_xlim(minn, maxx)
    ax.tick_params(
    axis='both',         # apply to both x & y axes
    which='major',       # only affect major ticks
    labelsize=20,        # font size of tick labels
    length=8,            # length of tick marks in points
    width=1.5            # width of the tick marks
    )
    ax.set_ylim(0, mayy)
    plt.tight_layout()
    if filename:
        fig.savefig(filename)
    plt.show()

# Porosity
plot_and_save(idx_porosity, 'crack porosity', 0, 0.5, 13, bins=100, filename='porosity_bmw_41.png')

# Water saturation
plot_and_save(idx_water, 'water saturation', 0, 1.0, 6, bins=100, filename='saturation_bmw_41.png')

/var/folders/5p/cy9rmpp11w7ddk5wt_dw0lk00000gp/T/ipykernel_81020/1282293129.py:25: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [ ]:
bb1 = blobs.reshape(180000, 3)
post_lab = ['vp', 'vs', 'rhob']
plot_1d_histograms(bb1, labels=post_lab, bins=100, savefig = "Bmw_fig2.png")

/var/folders/5p/cy9rmpp11w7ddk5wt_dw0lk00000gp/T/ipykernel_81020/964710158.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [10]:

def plot_2d_color_plots(samples, lab, savefig=None):
    """
    Plot 2D color maps (histograms) for selected pairs of MCMC parameters.
    
    Parameters:
      samples: numpy array of shape (N, ndim) with MCMC chain samples.
      lab:     list of parameter labels (length ndim).
      savefig: (Optional) Filename to save the figure (e.g., "my_2dplots.png").
    """
    # Extract individual variables from the samples.
    aspect_ratio = samples[:, 0]
    porosity = samples[:, 1]
    water_saturation = samples[:, 2]
    mineral_bulk_modulus = samples[:, 3]
    mineral_shear_modulus = samples[:, 4]
    mineral_density = samples[:, 5]

    # List of variables to plot.
    variables = [aspect_ratio, porosity, water_saturation, mineral_bulk_modulus,
                 mineral_shear_modulus, mineral_density]

    # Define a list of pair indices to plot.
    pairs = [
        (0, 1), (1, 2), (2, 3), (3, 4), (4, 5),  # First 5 pairs.
        (0, 2), (1, 3), (2, 4), (3, 5),          # Next 4 pairs.
        (0, 3), (1, 4), (2, 5),                   # Next 3 pairs.
        (0, 4), (1, 5),                          # Next 2 pairs.
        (0, 5), (1, 0)                           # Last 2 pairs to complete the grid.
    ]
    total_pairs = len(pairs)

    # Create a grid for the subplots.
    nrows = 5
    ncols = 3
    fig, axs = plt.subplots(nrows, ncols, figsize=(15, 15))
    
    plot_counter = 0
    for i in range(nrows):
        for j in range(ncols):
            if plot_counter < total_pairs:
                x_idx, y_idx = pairs[plot_counter]

                # Select the pair of variables to plot.
                x_var = variables[x_idx]
                y_var = variables[y_idx]

                # Compute the 2D histogram with 100 bins per axis.
                hist, x_edges, y_edges = np.histogram2d(x_var, y_var, bins=100)

                # Plot the 2D color map using imshow.
                im = axs[i, j].imshow(hist.T, extent=[x_edges[0], x_edges[-1], y_edges[0], y_edges[-1]],
                                        origin="lower", aspect="auto", cmap="viridis")
                axs[i, j].set_title(f"{lab[y_idx]} vs {lab[x_idx]}")
                axs[i, j].set_xlabel(lab[x_idx])
                axs[i, j].set_ylabel(lab[y_idx])
                
                plot_counter += 1
            else:
                axs[i, j].axis('off')
    
    fig.tight_layout()
    fig.colorbar(im, ax=axs, orientation='horizontal', fraction=0.02, pad=0.04)
    
    
    plt.show()
    if savefig:
        plt.savefig(savefig)
    

# Example usage:
labels = [
    "Aspect Ratio", 
    "Porosity", 
    "Water Saturation", 
    "Matrix Bulk Modulus (Pa)", 
    "Matrix Shear Modulus (Pa)", 
    "Matrix Density (kg/m³)"
]

# Assuming 'samples' is your flattened MCMC chain (numpy array) with 6 columns.
plot_2d_color_plots(samples, labels, savefig="Bmw_fig3.png")

/var/folders/5p/cy9rmpp11w7ddk5wt_dw0lk00000gp/T/ipykernel_81020/1721167122.py:65: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [11]:
flat_log_prob = sampler.get_log_prob(flat=True)
best_idx = np.argmax(flat_log_prob)
best_params = samples[best_idx]

def W_thickness(S_w, phi):
    return 8500*S_w*phi

print("rough estimate of water layer thickness (m):", W_thickness(best_params[2], best_params[1]))

rough estimate of water layer thickness (m): 268.7901621678736


In [12]:
def marginal_mode(x, bins=100):
    counts, edges = np.histogram(x, bins=bins)
    # find bin with max count, then take its center
    idx = np.argmax(counts)
    return 0.5*(edges[idx] + edges[idx+1])

phi_mode = marginal_mode(samples[:,1], bins=100)
S_w_mode = marginal_mode(samples[:,2], bins=100)
print("Histogram-mode phi:", phi_mode)
print("Histogram-mode S_w:", S_w_mode)
print("Mode-based thickness:", W_thickness(S_w_mode, phi_mode))

Histogram-mode phi: 0.3033484480404124
Histogram-mode S_w: 0.9449544527266045
Mode-based thickness: 2436.528966979688
